# Parallel Experiments with Persistent Disk Clones

This notebook demonstrates a correctness-first clone flow:
1. Create a base environment on a persistent disk.
2. Flush the disk and clone it while the base reservation is still running.
3. Wait for the clone operation to finish registering.
4. Run one experiment on the source disk and one on the cloned disk in parallel.
5. Compare results and timing.


In [ ]:
%pip install -e .. -q

In [ ]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

from gpu_dev import GpuDev

client = GpuDev()
RUN_ID = os.environ.get("GPU_DEV_EXPERIMENT_ID", time.strftime("%m%d%H%M%S"))
BASE_DISK = f"experiment-base-{RUN_ID}"
VARIANT_DISK = f"experiment-variant-{RUN_ID}"

print(f"SDK v{__import__('gpu_dev').__version__}")
print(f"Run ID:       {RUN_ID}")
print(f"Base disk:    {BASE_DISK}")
print(f"Variant disk: {VARIANT_DISK}")

In [ ]:
def require_ok(result, step):
    if result.exit_code != 0:
        raise RuntimeError(
            f"{step} failed with exit {result.exit_code}\n"
            f"STDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
        )


def parse_results(output):
    for line in output.splitlines():
        if line.startswith("RESULTS_JSON:"):
            return json.loads(line.removeprefix("RESULTS_JSON:"))
    raise RuntimeError(f"Training output did not contain RESULTS_JSON:\n{output}")


def run_training(sandbox, *, env):
    prefix = " ".join(f"{key}={value}" for key, value in env.items())
    started = time.time()
    result = sandbox.exec(f"{prefix} python3 /home/dev/train.py", timeout=120)
    train_time = time.time() - started
    require_ok(result, env["EXP_NAME"])
    return {
        "timings": {"train": train_time},
        "results": parse_results(result.stdout),
        "train_output": result.stdout,
    }

## Step 1: Create the Base Disk

Reserve one GPU with a persistent disk and write a small parameterized training script. The setup cell ends with `sync` so the file is flushed before the live snapshot starts.


In [ ]:
timings = {}
base = None
variant = None
outputs = {}
total_start = time.time()

started = time.time()
base = client.reserve(
    gpu_type="t4",
    gpu_count=1,
    hours=0.5,
    disk_name=BASE_DISK,
    name=f"base-{RUN_ID}",
    timeout_minutes=10,
    on_progress=True,
)
timings["reserve_base"] = time.time() - started

print(f"Reserved base in {timings['reserve_base']:.1f}s")
print(f"Disk: {base.disk_name}")
print(f"GPU:  {base.gpu_type} x{base.gpu_count}")

In [ ]:
TRAIN_SCRIPT = r"""set -euo pipefail
cat > /home/dev/train.py << 'SCRIPT'
import json
import os
import time

import torch
import torch.nn as nn


lr = float(os.environ.get("LR", "0.001"))
batch_size = int(os.environ.get("BATCH_SIZE", "64"))
epochs = int(os.environ.get("EPOCHS", "3"))
steps = int(os.environ.get("STEPS", "20"))
experiment = os.environ.get("EXP_NAME", "default")

print(f"Experiment: {experiment}")
print(f"Config: lr={lr}, batch_size={batch_size}, epochs={epochs}, steps={steps}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

torch.manual_seed(0)
model = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 10),
).cuda()

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()
results = {
    "experiment": experiment,
    "lr": lr,
    "batch_size": batch_size,
    "losses": [],
    "epoch_times": [],
}

for epoch in range(epochs):
    start = time.time()
    loss_sum = 0.0
    for _ in range(steps):
        x = torch.randn(batch_size, 3, 32, 32, device="cuda")
        y = torch.randint(0, 10, (batch_size,), device="cuda")
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    torch.cuda.synchronize()
    avg_loss = loss_sum / steps
    epoch_time = time.time() - start
    results["losses"].append(avg_loss)
    results["epoch_times"].append(epoch_time)
    print(f"  Epoch {epoch + 1}/{epochs}: loss={avg_loss:.4f} ({epoch_time:.2f}s)")

results["final_loss"] = results["losses"][-1]
results["avg_epoch_time"] = sum(results["epoch_times"]) / len(results["epoch_times"])

with open(f"/home/dev/results_{experiment}.json", "w") as f:
    json.dump(results, f)

print("RESULTS_JSON:" + json.dumps(results))
SCRIPT
chmod +x /home/dev/train.py
sync
python3 - <<'PY'
import torch
print(f"train.py ready; cuda={torch.cuda.is_available()}")
PY
"""

started = time.time()
result = base.exec(TRAIN_SCRIPT, timeout=120)
require_ok(result, "setup")
timings["setup"] = time.time() - started
print(result.stdout.strip())
print(f"Setup in {timings['setup']:.1f}s")

## Step 2: Clone the Disk

The base reservation stays alive, but the disk is not modified while the clone runs. The SDK waits for the clone operation to register; with the correctness-first Lambda behavior this means the source snapshot has completed.


In [ ]:
started = time.time()
client.clone_disk(BASE_DISK, VARIANT_DISK, timeout=900)
timings["clone"] = time.time() - started
print(f"Clone registered in {timings['clone']:.1f}s")

for disk in client.disks():
    if disk.name in {BASE_DISK, VARIANT_DISK}:
        print(f"  {disk.name:35s} {disk.size_gb}GB snapshots={disk.snapshot_count}")

## Step 3: Run Parallel Experiments

Run the high learning-rate experiment on the original reservation and reserve a second GPU with the cloned disk for the low learning-rate experiment.


In [ ]:
started = time.time()
variant = client.reserve(
    gpu_type="t4",
    gpu_count=1,
    hours=0.5,
    disk_name=VARIANT_DISK,
    name=f"variant-{RUN_ID}",
    timeout_minutes=10,
    on_progress=True,
)
timings["reserve_variant"] = time.time() - started
print(f"Reserved variant in {timings['reserve_variant']:.1f}s")

In [ ]:
experiments = {
    "high-lr": {
        "sandbox": base,
        "env": {
            "LR": "0.01",
            "BATCH_SIZE": "128",
            "EPOCHS": "3",
            "STEPS": "20",
            "EXP_NAME": "high_lr",
        },
    },
    "low-lr": {
        "sandbox": variant,
        "env": {
            "LR": "0.0001",
            "BATCH_SIZE": "32",
            "EPOCHS": "3",
            "STEPS": "20",
            "EXP_NAME": "low_lr",
        },
    },
}

started = time.time()
with ThreadPoolExecutor(max_workers=2) as pool:
    futures = {
        pool.submit(run_training, exp["sandbox"], env=exp["env"]): name
        for name, exp in experiments.items()
    }
    for future in as_completed(futures):
        name = futures[future]
        outputs[name] = future.result()
        print(f"{name} completed")

timings["parallel_train"] = time.time() - started
print(f"Parallel training completed in {timings['parallel_train']:.1f}s")

## Step 4: Compare Results

In [ ]:
high = outputs["high-lr"]["results"]
low = outputs["low-lr"]["results"]

print("=" * 60)
print(f"{'Metric':<25s} {'High LR':>15s} {'Low LR':>15s}")
print("=" * 60)
print(f"{'Learning Rate':<25s} {high['lr']:>15.4f} {low['lr']:>15.4f}")
print(f"{'Batch Size':<25s} {high['batch_size']:>15d} {low['batch_size']:>15d}")
print(f"{'Final Loss':<25s} {high['final_loss']:>15.4f} {low['final_loss']:>15.4f}")
print(f"{'Avg Epoch Time (s)':<25s} {high['avg_epoch_time']:>15.2f} {low['avg_epoch_time']:>15.2f}")
print()

print("Loss progression:")
for i in range(len(high["losses"])):
    print(f"  Epoch {i + 1}: high_lr={high['losses'][i]:.4f}  low_lr={low['losses'][i]:.4f}")

## Step 5: Timing Breakdown

In [ ]:
total_time = time.time() - total_start

print("Timing breakdown")
print("=" * 60)
for key, value in timings.items():
    print(f"{key:<20s} {value:>8.1f}s")
print("-" * 60)
print(f"{'total':<20s} {total_time:>8.1f}s")

## Cleanup

Cancel the reservations. The disks are preserved by default so you can inspect them or rerun from the same state.


In [ ]:
for label, sandbox in (("variant", variant), ("base", base)):
    if sandbox is None:
        continue
    try:
        sandbox.cancel()
        print(f"Cancelled {label}")
    except Exception as exc:
        print(f"Could not cancel {label}: {exc}")

print("Disks preserved:")
print(f"  {BASE_DISK}")
print(f"  {VARIANT_DISK}")
print("Delete later with client.delete_disk(...) when you no longer need them.")